<a href="https://colab.research.google.com/github/irenetobby/streamlit_fastapi_bank/blob/main/fastapi_bank_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pickle
from sklearn.tree import DecisionTreeClassifier
import numpy as np

# Create a dummy model for demonstration purposes
dummy_classifier = DecisionTreeClassifier()

# Fit the dummy classifier with some arbitrary data
X_dummy = np.array([[0.1, 0.2, 0.3, 0.4], [0.5, 0.6, 0.7, 0.8], [0.9, 1.0, 1.1, 1.2], [1.3, 1.4, 1.5, 1.6]])
y_dummy = np.array([0, 1, 0, 1])
dummy_classifier.fit(X_dummy, y_dummy)

# Save the dummy classifier to 'Classifier.pkl'
with open('Classifier.pkl', 'wb') as file:
    pickle.dump(dummy_classifier, file)

print("Dummy 'Classifier.pkl' created successfully.")

Dummy 'Classifier.pkl' created successfully.


A dummy `Classifier.pkl` has been created. You can now re-run the cell with the FastAPI application. Remember to replace this dummy file with your actual trained model for correct predictions.

In [12]:
from fastapi import FastAPI
import uvicorn
# from Banknotes import BankNote # Original line causing the error
from pydantic import BaseModel # Import BaseModel
import numpy as np
import pickle
import pandas as pd
import nest_asyncio # Import nest_asyncio
import threading # Import threading

nest_asyncio.apply() # Apply nest_asyncio at the top level to ensure it patches before any event loop starts

# Define the BankNote class as a Pydantic BaseModel
class BankNote(BaseModel):
    variance: float
    skewness: float
    curtosis: float
    entropy: float

# creating the instance
app = FastAPI()
pickle_in = open('Classifier.pkl', 'rb')
# make the classifier model from pickle
classifier = pickle.load(pickle_in)
# index route for opening the local host
@app.get('/')
def index():
    return {'message': 'Hello, Welcome'}
@app.get('/Welcome')
def get_name(name: str):
    return {'Welcome to sample Fast API Deployment': f'{name}'}
@app.post('/predict')
def predict_banknote(data: BankNote): # the class created in BankNotes.py, data created in JSON form, now convert to dictionary
    data = data.dict()
    variance = data['variance']
    skewness = data['skewness']
    curtosis = data['curtosis']
    entropy = data['entropy']

    prediction = classifier.predict([[variance, skewness, curtosis, entropy]])
    if prediction[0] > 0.5:
        prediction = 'The note is Fake'
    else:
        prediction = 'The note is original'
    return {'prediction': prediction}

def run_uvicorn():
    uvicorn.run(app, host="127.0.0.1", port=8000)

if __name__ == "__main__":
    # Run Uvicorn in a separate thread to avoid event loop conflicts in Colab
    thread = threading.Thread(target=run_uvicorn)
    thread.start()
    print("FastAPI app is running in a separate thread on http://127.0.0.1:8000")
    print("You might need to use ngrok or similar for external access in Colab.")

FastAPI app is running in a separate thread on http://127.0.0.1:8000
You might need to use ngrok or similar for external access in Colab.
